# 05-06 Configure: 런타임에 체인 설정 바꾸기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `configurable_fields`로 체인 실행 시점에 모델 이름, temperature 등 파라미터를 동적으로 변경할 수 있어요
- `configurable_alternatives`로 런타임에 완전히 다른 모델이나 프롬프트를 교체할 수 있어요
- `with_config()`로 특정 설정을 미리 적용한 체인 버전을 만들고 재사용할 수 있어요
- 두 방식의 차이를 구분하고 상황에 맞게 선택할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `ChatOpenAI` 모델 초기화와 파라미터 설정 방법 (Ch01)
- LCEL 파이프 연산자(`|`) 사용법 (Ch05-01~03)
- `RunnableConfig` 기본 개념 (Ch05-03)

---

체인을 한 번 만들어 놓고, 호출할 때마다 다양한 옵션을 동적으로 바꾸는 방법을 배워요. 코드를 수정하지 않고 `config` 딕셔너리만 바꿔서 다른 모델, 다른 프롬프트를 사용할 수 있어요.

**두 가지 방식:**

| 방식 | 변경 대상 | 비유 |
|------|----------|------|
| `configurable_fields` | Runnable의 **파라미터 값** | TV 볼륨·채널 조절 (리모컨 버튼) |
| `configurable_alternatives` | **Runnable 객체** 자체 | TV 자체를 다른 TV로 교체 |

> 🔑 **핵심 개념**: 설정 가능 필드(`ConfigurableField`)는 기존 Runnable의 "파라미터"를 런타임에 바꿀 수 있게 해요. 체인을 다시 만들 필요 없이 `config` 값만 전달하면 돼요.

아래 다이어그램은 두 방식이 체인 실행에 어떻게 영향을 주는지 보여줘요.

```mermaid
flowchart LR
    subgraph 체인["체인 (한 번 정의)"]
        P["프롬프트"]:::process
        M["모델"]:::process
        P --> M
    end

    CF["configurable_fields<br/>파라미터 값 변경<br/>예: temperature=0.9"]:::input
    CA["configurable_alternatives<br/>Runnable 교체<br/>예: GPT-4o → GPT-3.5"]:::input

    CF -->|"config 전달"| M
    CA -->|"config 전달"| M

    M --> R1["응답 A"]:::output
    M --> R2["응답 B"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI

# 기본 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

## 1. configurable_fields — 파라미터를 동적으로 변경하기

`configurable_fields`를 사용하면 모델의 특정 필드를 런타임에 동적으로 변경할 수 있어요.

예를 들어, `model_name` 파라미터를 동적으로 설정하면 호출 시점에 다른 모델을 사용할 수 있어요.

### 동작 원리

1. `ConfigurableField(id="키이름")`으로 어떤 파라미터를 동적으로 만들지 선언해요
2. `invoke()` 시 `config={"configurable": {"키이름": "새값"}}`으로 값을 전달해요
3. 해당 파라미터가 전달된 값으로 교체되어 실행돼요

### 설정 적용 방법 두 가지

```python
# 방법 1: invoke() 시 매번 config 전달
model.invoke("질문", config={"configurable": {"gpt_version": "gpt-4o-mini"}})

# 방법 2: with_config()로 설정을 미리 고정 후 재사용
gpt4o_model = model.with_config(configurable={"gpt_version": "gpt-4o-mini"})
gpt4o_model.invoke("질문")
```

> 💡 **실무 팁**: A/B 테스트나 사용자 요금제별로 다른 모델을 적용할 때 유용해요. 하나의 체인 코드베이스에서 `config` 값만 바꿔 여러 모델을 관리할 수 있어요.

> 🎯 **강의 포인트**: `configurable_fields`는 값을 바꾸고, `configurable_alternatives`는 전체 컴포넌트를 교체해요. 이 차이를 먼저 강조하면 이후 섹션에서 혼동이 줄어요.

In [3]:
# ============================================================
# TODO: configurable_fields()로 model_name을 동적으로 변경 가능하도록 설정하세요
# 힌트: ChatOpenAI(...).configurable_fields(model_name=ConfigurableField(id="gpt_version", ...))
# 예상 결과: invoke() 시 config={"configurable": {"gpt_version": "..."}}으로 모델 교체
# ============================================================

# ---------------------------------------------------
# configurable_fields: 파라미터 값을 런타임에 동적으로 변경
# ---------------------------------------------------

# 1단계: configurable_fields로 model_name 필드를 동적 설정 가능하게 만들기
# ConfigurableField: 동적으로 변경할 필드 정의
#   - id: invoke() 시 config에서 사용하는 키 이름
#   - name: 사람이 읽기 쉬운 필드 이름
#   - description: 필드 설명
configurable_model = ChatOpenAI(temperature=0).configurable_fields(
    model_name=ConfigurableField(
        id="gpt_version",  # 설정 시 사용할 키 이름
        name="Version of GPT",  # 필드 이름
        description="사용할 GPT 모델 버전. 예: gpt-4o, gpt-4o-mini, gpt-3.5-turbo",
    )
)

print("=" * 60)
print("✅ configurable_fields 설정 완료")
print("=" * 60)
print("이제 invoke() 시 config로 모델을 동적으로 변경할 수 있습니다.")


✅ configurable_fields 설정 완료
이제 invoke() 시 config로 모델을 동적으로 변경할 수 있습니다.


### 1.1 config 없이 기본값으로 실행하기

`configurable_fields`를 설정한 뒤 `config`를 전달하지 않으면, 모델 초기화 시 지정한 기본값이 그대로 사용돼요. 아래에서 기본 모델과 config로 모델을 변경한 결과를 비교해 볼게요.

In [4]:
# 기본 모델로 실행 (설정 없이)
result_default = configurable_model.invoke("대한민국의 수도는 어디야?")

print("=" * 60)
print("📥 기본 모델 실행")
print("=" * 60)
print(f"모델: 기본값")
print(f"답변: {result_default.content}")
print()
print(f"사용된 모델: {result_default.response_metadata.get('model_name', 'N/A')}")


📥 기본 모델 실행
모델: 기본값
답변: 대한민국의 수도는 서울이야.

사용된 모델: gpt-3.5-turbo-0125


### 1.2 config로 모델 변경하여 실행하기

이번에는 `config={"configurable": {"gpt_version": "모델명"}}`을 전달하여 다른 모델로 실행해 볼게요. 같은 `configurable_model` 객체를 사용하지만 `config` 값에 따라 실제 호출되는 모델이 달라져요.

In [5]:
# config로 모델 변경하여 실행
# 
# config={"configurable": {"키": "값"}} 형식으로 동적 설정

result_gpt35 = configurable_model.invoke(
    "대한민국의 수도는 어디야?",
    config={"configurable": {"gpt_version": "gpt-3.5-turbo"}},
)

print("=" * 60)
print("📥 gpt-3.5-turbo 모델로 실행")
print("=" * 60)
print(f"설정: gpt_version='gpt-3.5-turbo'")
print(f"답변: {result_gpt35.content}")
print()
print(f"사용된 모델: {result_gpt35.response_metadata.get('model_name', 'N/A')}")


📥 gpt-3.5-turbo 모델로 실행
설정: gpt_version='gpt-3.5-turbo'
답변: 대한민국의 수도는 서울이야.

사용된 모델: gpt-3.5-turbo-0125


In [6]:
# 다른 모델로 실행
result_gpt4o = configurable_model.invoke(
    "대한민국의 수도는 어디야?",
    config={"configurable": {"gpt_version": "gpt-4o-mini"}},
)

print("=" * 60)
print("📥 gpt-4o-mini 모델로 실행")
print("=" * 60)
print(f"설정: gpt_version='gpt-4o-mini'")
print(f"답변: {result_gpt4o.content}")
print()
print(f"사용된 모델: {result_gpt4o.response_metadata.get('model_name', 'N/A')}")


📥 gpt-4o-mini 모델로 실행
설정: gpt_version='gpt-4o-mini'
답변: 대한민국의 수도는 서울입니다.

사용된 모델: gpt-4o-mini-2024-07-18


### 1.3 with_config()로 설정 미리 고정하기

매번 `config`를 전달하는 대신, `with_config()`를 사용하면 특정 설정이 미리 적용된 새로운 Runnable을 만들 수 있어요. 자주 쓰는 설정 조합을 변수에 저장해 두면 편리해요.

이 방식은 **사전 설정(Pre-configuration)** 패턴이에요:

- `invoke()` 시 config 전달 = 호출할 때마다 설정 지정
- `with_config()` = 설정을 미리 고정한 새 Runnable 생성

> 💡 **실무 팁**: 프로덕션 코드에서는 `with_config()`로 "고급 모델용 체인", "저비용 모델용 체인" 등을 미리 만들어 두고, API 엔드포인트별로 다른 체인을 사용하는 패턴이 흔해요.

In [7]:
# with_config()로 설정을 미리 지정
gpt4o_model = configurable_model.with_config(
    configurable={"gpt_version": "gpt-4o-mini"}
)

# 설정이 적용된 모델로 실행
result = gpt4o_model.invoke("대한민국의 수도는 어디야?")

print("=" * 60)
print("📥 with_config() 사용 예제")
print("=" * 60)
print(f"답변: {result.content}")
print()
print("💡 with_config()로 설정을 미리 지정하면")
print("   매번 config를 전달할 필요 없이 재사용 가능")


📥 with_config() 사용 예제
답변: 대한민국의 수도는 서울입니다.

💡 with_config()로 설정을 미리 지정하면
   매번 config를 전달할 필요 없이 재사용 가능


## 2. configurable_alternatives — Runnable 자체를 교체하기

`configurable_alternatives`를 사용하면 런타임에 여러 Runnable 중 하나를 선택할 수 있어요.

`configurable_fields`가 **파라미터 값**을 바꾸는 것이라면, `configurable_alternatives`는 **Runnable 객체 자체**를 통째로 바꾸는 것이에요.

### 주요 구성 요소

| 구성 요소 | 설명 |
|-----------|------|
| `ConfigurableField(id="llm")` | config에서 사용할 설정 키 이름 |
| `default_key="openai_mini"` | config 없이 실행할 때 사용되는 기본 Runnable의 키 |
| 키워드 인자 (`openai=ChatOpenAI(...)`) | 대안으로 선택할 수 있는 Runnable 목록 |

> ⚠️ **주의**: `default_key`는 `configurable_alternatives`에서 반드시 지정해야 해요. 이 키가 설정 없이 체인을 실행할 때 사용되는 기본 Runnable을 결정해요. 빠뜨리면 기본값 없이 오류가 발생해요.

> 🔑 **핵심 개념**: `configurable_alternatives`의 키워드 인자 이름(예: `openai`)이 `with_config(configurable={"llm": "openai"})`에서 사용하는 값과 일치해야 해요.

In [8]:
# ============================================================
# TODO: configurable_alternatives()로 여러 모델을 런타임에 선택 가능하도록 설정하세요
# 힌트: ChatOpenAI(...).configurable_alternatives(ConfigurableField(id="llm"), default_key="openai_mini", openai=...)
# 예상 결과: with_config(configurable={"llm": "openai"})로 모델 교체
# ============================================================

# ---------------------------------------------------
# configurable_alternatives: Runnable 객체 자체를 런타임에 교체
# ---------------------------------------------------

# 1단계: 기본 모델 설정 및 대안 모델 정의
# configurable_alternatives: 필드 값이 아닌 Runnable 객체 전체를 교체
# - default_key: 설정 없이 실행할 때 사용하는 기본 키 (기본 객체 자체를 가리킴)
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini").configurable_alternatives(
    ConfigurableField(id="llm"),  # 설정 키
    default_key="openai_mini",  # 기본 키 (= 위의 ChatOpenAI 객체를 가리킴)
    # 대안 모델들 정의 (키워드 인자로 지정)
    # ⚠️ default_key와 동일한 키("openai_mini")는 등록하면 안 됨 (중복 오류 발생)
    openai=ChatOpenAI(model_name="gpt-4o"),
    openai_turbo=ChatOpenAI(model_name="gpt-3.5-turbo"),
)

# 2단계: 프롬프트와 체인 생성
prompt = ChatPromptTemplate.from_template("{topic}에 대해 간단히 설명해주세요.")
chain = prompt | llm

print("=" * 60)
print("✅ configurable_alternatives 설정 완료")
print("=" * 60)
print("사용 가능한 모델:")
print("  - openai_mini (기본)")
print("  - openai")
print("  - openai_turbo")

✅ configurable_alternatives 설정 완료
사용 가능한 모델:
  - openai_mini (기본)
  - openai
  - openai_turbo


### 2.1 기본 모델과 대안 모델로 실행 비교

config 없이 실행하면 `default_key`에 매핑된 기본 Runnable(gpt-4o-mini)이 사용돼요. `with_config(configurable={"llm": "openai"})`를 전달하면 키워드 인자 `openai`에 매핑된 gpt-4o 모델로 교체돼요.

In [9]:
# 기본 모델로 실행
result_default = chain.invoke({"topic": "인공지능"})

print("=" * 60)
print("📥 기본 모델 (openai_mini) 실행")
print("=" * 60)
print(result_default.content[:200] + "...")


📥 기본 모델 (openai_mini) 실행
인공지능(AI, Artificial Intelligence)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해, 언어 처리 등의 작업을 수행할 수 있도록 하는 기술입니다. AI는 머신러닝(기계 학습), 딥러닝(심층 학습), 자연어 처리(NLP) 등 다양한 하위 분야로 나뉘며, 데이터 분석과 패턴 인식을 통해 스스로 개선하고 발전할 수...


In [10]:
# 다른 모델로 실행
result_gpt4o = chain.with_config(configurable={"llm": "openai"}).invoke(
    {"topic": "인공지능"}
)

print("=" * 60)
print("📥 gpt-4o 모델로 실행")
print("=" * 60)
print(result_gpt4o.content[:200] + "...")


📥 gpt-4o 모델로 실행
인공지능(AI)은 기계가 인간과 유사한 지능적 작업을 수행할 수 있도록 하는 기술과 시스템을 말합니다. 이는 주로 컴퓨터 시스템이 학습, 추론, 문제 해결, 지각 및 언어 이해와 같은 인간의 지적 과정을 모방하는 것을 목표로 합니다. AI는 다양한 분야에서 응용되며, 그 중 몇 가지 주요 분야는 다음과 같습니다:

1. **기계 학습(ML)**: AI의 하...


## 3. 프롬프트 대안 설정

`configurable_alternatives`는 모델뿐 아니라 **프롬프트**에도 적용할 수 있어요. 같은 입력 변수(`{country}`)를 사용하는 여러 프롬프트 템플릿을 정의해 두고, 런타임에 원하는 프롬프트를 선택하는 패턴이에요.

> 🎯 **강의 포인트**: 프롬프트 대안은 "같은 입력 데이터, 다른 질문"이 필요한 상황에 유용해요. 예를 들어, 하나의 국가 데이터로 수도·면적·인구 등 다양한 정보를 질문할 수 있어요.

In [11]:
# ============================================================
# TODO: 프롬프트에도 configurable_alternatives를 설정하고 런타임에 교체하세요
# 힌트: ChatPromptTemplate.from_template(...).configurable_alternatives(ConfigurableField(id="prompt"), default_key="capital", area=..., population=...)
# 예상 결과: with_config(configurable={"prompt": "area"})로 프롬프트 교체
# ============================================================

# ---------------------------------------------------
# 프롬프트에 configurable_alternatives 적용
# ---------------------------------------------------

llm = ChatOpenAI(temperature=0)

# 1단계: 여러 프롬프트 템플릿 정의 및 configurable_alternatives 적용
# - default_key="capital": 설정 없이 실행 시 수도 프롬프트 사용 (= 위의 기본 템플릿)
# ⚠️ default_key와 동일한 키("capital")는 키워드 인자로 등록하면 안 됨 (중복 오류 발생)
prompt = ChatPromptTemplate.from_template(
    "{country}의 수도는 어디야?"  # 기본 프롬프트 (default_key="capital"이 이 객체를 가리킴)
).configurable_alternatives(
    ConfigurableField(id="prompt"),
    default_key="capital",  # 기본 키
    # 대안 프롬프트들 (키워드 인자)
    area=ChatPromptTemplate.from_template("{country}의 면적은 얼마야?"),
    population=ChatPromptTemplate.from_template("{country}의 인구는 얼마야?"),
    language=ChatPromptTemplate.from_template("{country}의 공용어는 무엇인가요?"),
)

chain = prompt | llm

print("=" * 60)
print("✅ 프롬프트 대안 설정 완료")
print("=" * 60)

✅ 프롬프트 대안 설정 완료


아래에서 기본 프롬프트(수도)와 다른 프롬프트(면적, 인구)로 같은 국가를 질문해 볼게요. `config`에 전달하는 `prompt` 값만 바뀌는 것에 주목하세요.

In [12]:
# 기본 프롬프트로 실행
result_capital = chain.invoke({"country": "대한민국"})

print("=" * 60)
print("📥 기본 프롬프트 (수도) 실행")
print("=" * 60)
print(result_capital.content)


📥 기본 프롬프트 (수도) 실행
대한민국의 수도는 서울이야.


In [13]:
# 다른 프롬프트로 실행
result_area = chain.with_config(configurable={"prompt": "area"}).invoke(
    {"country": "대한민국"}
)

print("=" * 60)
print("📥 면적 프롬프트로 실행")
print("=" * 60)
print(result_area.content)


📥 면적 프롬프트로 실행
대한민국의 총 면적은 약 100,363km² 입니다.


In [14]:
# 인구 프롬프트로 실행
result_population = chain.with_config(configurable={"prompt": "population"}).invoke(
    {"country": "대한민국"}
)

print("=" * 60)
print("📥 인구 프롬프트로 실행")
print("=" * 60)
print(result_population.content)


📥 인구 프롬프트로 실행
2021년 7월 기준 대한민국의 인구는 약 5천 133만 명입니다.


## 4. 프롬프트와 LLM 모두 변경하기

프롬프트와 LLM에 각각 `configurable_alternatives`를 설정하면, 런타임에 **둘 다 동시에** 변경할 수 있어요.

하나의 `config` 딕셔너리 안에 `"prompt"` 키와 `"llm"` 키를 함께 전달하면 돼요:

```python
chain.with_config(configurable={"prompt": "founder", "llm": "openai"})
```

> 💡 **실무 팁**: 이 패턴은 다중 모델 지원, 사용자별 설정 커스터마이징에 유용해요. 예를 들어, 유료 사용자에게는 GPT-4o + 상세 프롬프트를, 무료 사용자에게는 GPT-3.5 + 간략 프롬프트를 제공할 수 있어요.

In [15]:
# ============================================================
# TODO: 프롬프트와 LLM 모두 configurable_alternatives로 설정하고 동시에 교체하세요
# 힌트: with_config(configurable={"prompt": "founder", "llm": "openai"})
# 예상 결과: 애플 창립자 질문을 gpt-4o로 답변
# ============================================================

# ---------------------------------------------------
# 프롬프트 + LLM 동시 configurable_alternatives 설정
# ---------------------------------------------------

# 1단계: LLM configurable_alternatives 설정
# ⚠️ default_key와 동일한 키("openai_mini")는 키워드 인자로 등록하면 안 됨
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini").configurable_alternatives(
    ConfigurableField(id="llm"),
    default_key="openai_mini",
    openai=ChatOpenAI(model_name="gpt-4o"),
)

# 2단계: 프롬프트 configurable_alternatives 설정
# ⚠️ default_key와 동일한 키("description")는 키워드 인자로 등록하면 안 됨
prompt = ChatPromptTemplate.from_template(
    "{company}에 대해서 20자 이내로 설명해 줘."  # 기본 프롬프트 (default_key="description"이 이 객체를 가리킴)
).configurable_alternatives(
    ConfigurableField(id="prompt"),
    default_key="description",
    founder=ChatPromptTemplate.from_template("{company}의 창립자는 누구인가요?"),
    competitor=ChatPromptTemplate.from_template("{company}의 경쟁사는 누구인가요?"),
)

chain = prompt | llm

print("=" * 60)
print("✅ 프롬프트와 LLM 모두 설정 완료")
print("=" * 60)

✅ 프롬프트와 LLM 모두 설정 완료


In [16]:
# 프롬프트와 LLM 모두 변경하여 실행
result = chain.with_config(
    configurable={"prompt": "founder", "llm": "openai"}
).invoke({"company": "애플"})

print("=" * 60)
print("📥 프롬프트(founder) + LLM(gpt-4o) 실행")
print("=" * 60)
print(result.content)


📥 프롬프트(founder) + LLM(gpt-4o) 실행
애플(Apple Inc.)의 창립자는 스티브 잡스(Steve Jobs), 스티브 워즈니악(Steve Wozniak), 그리고 로널드 웨인(Ronald Wayne)입니다. 회사는 1976년에 설립되었습니다.


## 5. 설정 저장 및 재사용

`with_config()`로 특정 설정을 적용한 체인을 별도의 변수에 저장하면, 이후에 `config` 없이 바로 `invoke()`할 수 있어요. 여러 설정 조합을 미리 만들어 두면 코드가 깔끔해져요.

> 🎯 **강의 포인트**: `with_config()`는 원본 체인을 변경하지 않고 새로운 체인을 반환해요. 원본과 설정이 적용된 체인을 동시에 사용할 수 있어요.

In [17]:
# 특정 설정으로 체인 저장
competitor_chain = chain.with_config(
    configurable={"llm": "openai", "prompt": "competitor"}
)

# 저장된 체인으로 실행
result = competitor_chain.invoke({"company": "애플"})

print("=" * 60)
print("📥 저장된 체인으로 실행")
print("=" * 60)
print(result.content)
print()
print("💡 특정 설정을 자주 사용할 때 유용")


📥 저장된 체인으로 실행
애플의 주요 경쟁사는 다양한 분야에서 활동하며 각각 다른 제품과 서비스를 제공합니다. 주요 경쟁사로는 다음과 같은 기업들이 있습니다:

1. **삼성전자**: 스마트폰, 태블릿, 스마트워치 등 모바일 기기 분야에서 애플의 직접적인 경쟁사입니다.

2. **구글**: 안드로이드 운영체제를 통해 스마트폰 분야에서 경쟁하며, 구글 픽셀 시리즈와 같은 자체 하드웨어 제품도 출시하고 있습니다. 또한, 클라우드 서비스나 인공지능 분야에서도 경쟁합니다.

3. **마이크로소프트**: 주로 소프트웨어(Windows)와 하드웨어(서피스 라인업)에서 경쟁합니다. 또한, 클라우드 컴퓨팅(Azure) 및 생산성 소프트웨어(Office 365) 분야에서도 경쟁합니다.

4. **아마존**: 주로 클라우드 컴퓨팅(AWS) 및 스마트 홈 기기 분야에서 경쟁합니다.

5. **화웨이**: 주로 스마트폰 및 통신장비 분야에서 경쟁합니다.

6. **샤오미**: 상대적으로 저렴한 가격대의 스마트폰, 스마트 홈 기기, 웨어러블 장치 등에서 애플과 경쟁합니다.

이 외에도 다양한 소프트웨어 및 하드웨어 분야에서 애플과 간접적으로 경쟁하는 기업들이 많습니다.

💡 특정 설정을 자주 사용할 때 유용


## 정리

### configurable_fields vs configurable_alternatives 비교

| 항목 | `configurable_fields` | `configurable_alternatives` |
|------|-----------------------|-----------------------------|
| 변경 대상 | Runnable의 **파라미터 값** | **Runnable 객체** 자체 |
| 사용 예시 | model_name, temperature 변경 | 모델 교체, 프롬프트 교체 |
| 설정 방법 | `ConfigurableField(id=...)` | `ConfigurableField(id=...) + 키워드 인자` |
| 기본값 지정 | 객체 초기화 시 파라미터 | `default_key` |
| 비유 | TV 볼륨 조절 | TV 자체를 교체 |

### 설정 적용 방법 비교

| 방법 | 코드 | 특징 |
|------|------|------|
| 매번 전달 | `chain.invoke(input, config={"configurable": {...}})` | 호출마다 다른 설정 가능 |
| 미리 고정 | `chain_v2 = chain.with_config(configurable={...})` | 재사용에 편리 |

### 이번 노트북에서 배운 핵심

1. **configurable_fields**: 기존 Runnable의 파라미터를 런타임에 변경해요
2. **configurable_alternatives**: Runnable 객체 자체를 런타임에 교체해요
3. **with_config()**: 자주 쓰는 설정을 미리 고정한 새 Runnable을 만들어요
4. 프롬프트와 LLM에 동시에 적용하면 하나의 체인으로 다양한 조합을 관리할 수 있어요

### 다음 노트북 예고

다음 노트북(`07-RunnableWithMessageHistory`)에서는 체인에 대화 기록을 추가하여 이전 대화 맥락을 기억하는 `RunnableWithMessageHistory` 사용법을 배워요. 세션별 대화 기록을 관리하는 챗봇 패턴을 살펴볼게요.